# 3. Training: FPN + U-Net + Swin-Transformer on ARCADE

Train all three models with 5-fold cross-validation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import json
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Setup Paths & Config

In [ ]:
# Paths
DATA_DIR = Path('D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments/paper_implementations/vessel_segmentation/data')
RESULTS_DIR = Path('D:/MPHIL_CODES/MPHIL_MAIN_REPO/experiments/paper_implementations/vessel_segmentation/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Training config
CONFIG = {
    'image_size': 384,
    'batch_size': 8,
    'num_epochs': 50,
    'learning_rate': 1e-3,
    'early_stopping_patience': 10,
    'num_folds': 5
}

print(f"Data dir: {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"\nTraining config: {CONFIG}")

## Custom Dataset

In [ ]:
class VesselSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, image_size=384, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size
        self.augment = augment
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img = Image.open(self.image_paths[idx]).convert('L')
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        img = np.array(img, dtype=np.float32) / 255.0
        
        # Load mask
        mask = Image.open(self.mask_paths[idx]).convert('L')
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)
        mask = np.array(mask, dtype=np.float32) / 255.0
        
        # Augmentation (rotation)
        if self.augment and np.random.rand() < 0.5:
            k = np.random.randint(1, 4)
            img = np.rot90(img, k).copy()
            mask = np.rot90(mask, k).copy()
        
        # Convert to tensors
        img = torch.from_numpy(img).unsqueeze(0).float()  # [1, H, W]
        mask = torch.from_numpy(mask).unsqueeze(0).float()  # [1, H, W]
        
        return img, mask

print("✓ Dataset class defined")

## Dice Loss

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice

print("✓ Dice loss defined")

## Load Data & Prepare Folds

In [ ]:
# Load dataset index
df_index = pd.read_csv(DATA_DIR / 'dataset_index.csv')

# Load fold split
with open(DATA_DIR / 'fold_split.json', 'r') as f:
    fold_splits = json.load(f)

print(f"Total images: {len(df_index)}")
print(f"Folds: {len(fold_splits)}")

# Prepare image and mask paths
image_paths = df_index['image_path'].values
mask_paths = df_index['mask_path'].values

print("\nFold split:")
for fold in fold_splits:
    print(f"  Fold {fold['fold']}: {fold['train_count']} train, {fold['val_count']} val")

## Training Function

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(train_loader, desc="Training", unit="batch")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{total_loss / (pbar.n + 1):.4f}'})
    
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_dice = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validating", unit="batch")
        for images, masks in pbar:
            images, masks = images.to(device), masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
            
            # Calculate Dice
            pred_binary = (torch.sigmoid(outputs) > 0.5).float()
            intersection = (pred_binary * masks).sum()
            union = pred_binary.sum() + masks.sum()
            dice = (2.0 * intersection) / (union + 1e-7)
            all_dice.append(dice.cpu().item())
            
            pbar.set_postfix({'loss': f'{total_loss / (pbar.n + 1):.4f}', 'dice': f'{np.mean(all_dice):.4f}'})
    
    return total_loss / len(val_loader), np.mean(all_dice)

print("✓ Training functions defined")

## Train All Models (5-Fold CV)

In [ ]:
# Import model architectures from previous notebook
# (In practice, these should be in a separate models.py file)

# For now, we'll define them inline
from torchvision import models
import timm

# [Include model definitions from 02_model_architectures.ipynb]

print("Models to train:")
print("  1. U-Net with SE-ResNet18")
print("  2. FPN with SE-ResNet18")
print("  3. Swin-Transformer")

In [ ]:
# Initialize results storage
all_results = {}

# Model configurations
models_config = {
    'unet': {'class_name': 'UNetWithSEResNet18'},
    'fpn': {'class_name': 'FPNWithSEResNet18'},
    'swin': {'class_name': 'SwinTransformerSegmentation'}
}

# Train for each fold
for fold_idx in range(CONFIG['num_folds']):
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx + 1}/{CONFIG['num_folds']}")
    print(f"{'='*80}")
    
    fold_info = fold_splits[fold_idx]
    train_indices = fold_info['train_indices']
    val_indices = fold_info['val_indices']
    
    # Create datasets
    train_dataset = VesselSegmentationDataset(
        image_paths[train_indices],
        mask_paths[train_indices],
        image_size=CONFIG['image_size'],
        augment=True
    )
    val_dataset = VesselSegmentationDataset(
        image_paths[val_indices],
        mask_paths[val_indices],
        image_size=CONFIG['image_size'],
        augment=False
    )
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Train each model
    for model_name, model_config in models_config.items():
        print(f"\n[{model_name.upper()}] Training...")
        
        # TODO: Create model instance based on model_name
        # model = create_model(model_name)
        # model = model.to(device)
        
        # Setup training
        # optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
        # criterion = DiceLoss()
        # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
        
        # TODO: Training loop
        
        print(f"  Model training placeholder for fold {fold_idx}")

print("\n✓ Training framework set up")

## Results Summary

In [ ]:
print("\n" + "="*80)
print("TRAINING WILL BEGIN HERE")
print("="*80)
print("\nThis notebook provides the training framework for:")
print("  ✓ 5-fold cross-validation setup")
print("  ✓ Data loading and augmentation")
print("  ✓ Training and validation loops")
print("  ✓ Dice loss and metrics computation")
print("\nNext step: Complete model implementations and run training!")